<a href="https://colab.research.google.com/github/deepthivj-aiml/5-Day-AI-Agents-Intensive-course-Google/blob/main/Copy_of_Fork_of_AutoEvalAI_AI_Powered_Second_Hand_Vehicle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# ==============================================
# 🔧 INSTALL DEPENDENCIES
# ==============================================
!pip install -q gradio pillow google-generativeai langchain langchain-community faiss-cpu pypdf requests langchain-text-splitters

In [5]:
import os, io, re, requests, tempfile
import gradio as gr
from PIL import Image, ImageEnhance, ImageFilter
import google.generativeai as genai

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

from google.colab import userdata

# ==============================================
# 🔐 GEMINI API KEY (Colab Secrets)
# ==============================================
GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
genai.configure(api_key=GOOGLE_API_KEY)
MODEL_NAME = "gemini-2.5-flash-lite"
model = genai.GenerativeModel(MODEL_NAME)

# ==============================================
# 📚 REAL, WORKING PDF URLs (VERIFIED)
# ==============================================
PDF_URLS = [
    "https://www.congressionalfcu.org/docs/default-source/PDFs/usedcartestdrive.pdf",
    "https://www.chris-fix.com/upload/How%20to%20Inspect%20a%20Used%20Car%20Checklist%20%20FULL.pdf",
    "https://miguelsgarage.com/wp-content/uploads/2020/04/BUYIT-Checklist.pdf",
    "https://www.cwcac.org/wp-content/uploads/2024/11/How-to-Inspect-Used-Car-Checklist-WM.pdf",
    "https://www.lincoln.com/cmslibs/content/dam/brand_lincoln/en_us/brand/cpo/pdf/LCPO01015_FMFL1388000A_L_VIC_NCR_R02.pdf"
]

# ==============================================
# 📥 DOWNLOAD PDFs
# ==============================================
pdf_files = []
tmp_dir = tempfile.mkdtemp()

for i, url in enumerate(PDF_URLS):
    r = requests.get(url, timeout=30)
    path = os.path.join(tmp_dir, f"doc_{i}.pdf")
    with open(path, "wb") as f:
        f.write(r.content)
    pdf_files.append(path)

# ==============================================
# 🧠 BUILD RAG VECTOR STORE
# ==============================================
docs = []
for pdf in pdf_files:
    loader = PyPDFLoader(pdf)
    docs.extend(loader.load())

splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
chunks = splitter.split_documents(docs)

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embeddings)

# ==============================================
# 🖼 IMAGE ENHANCEMENT
# ==============================================
def enhance(img):
    img = img.convert("RGB")
    img = img.filter(ImageFilter.SHARPEN)
    img = ImageEnhance.Contrast(img).enhance(1.2)
    img = ImageEnhance.Color(img).enhance(1.1)
    return img

# ==============================================
# 🔎 GEMINI CALL
# ==============================================
def gemini(prompt, img=None):
    if img:
        return model.generate_content([prompt, img]).text
    return model.generate_content(prompt).text

# ==============================================
# 🚗 VEHICLE DETAILS EXTRACTION (LLM)
# ==============================================
def extract_vehicle_details(img):
    prompt = """
    Identify vehicle details from the image.
    Return strictly in JSON:
    {
      "Type": "",
      "Make": "",
      "Model": "",
      "Color": ""
    }
    """
    text = gemini(prompt, img)
    match = re.search(r"\{.*\}", text, re.S)
    return eval(match.group()) if match else {}

# ==============================================
# 🗓 MODEL YEAR INFERENCE (VISION + RAG)
# ==============================================
def infer_model_year(vehicle, img):
    query = f"""
    Based on inspection manuals and vehicle design evolution,
    infer the model year range for:
    Make: {vehicle.get('Make')}
    Model: {vehicle.get('Model')}
    Color: {vehicle.get('Color')}
    """
    docs = vectorstore.similarity_search(query, k=3)
    context = "\n".join(d.page_content for d in docs)

    prompt = f"""
    Context:
    {context}

    Vehicle image visible features include headlights, grille, body lines.
    Estimate the most likely model year (single year).
    """
    year = gemini(prompt, img)
    match = re.search(r"(19|20)\d{2}", year)
    return match.group() if match else "Unknown"

# ==============================================
# ⭐ RATING EMOJI
# ==============================================
def emoji(r): return "🤩" if r>=8 else "🙂" if r>=6 else "😐" if r>=4 else "🙁"

# ==============================================
# 🔍 PART INSPECTION
# ==============================================
def inspect_part(title, img):
    prompt = f"""
    Inspect the {title} for damage, wear, safety risks.
    Give rating 1–10 and short explanation.
    """
    text = gemini(prompt, img)
    match = re.search(r"(\d+(\.\d+)?)", text)
    rating = float(match.group()) if match else 6
    return rating, text

# ==============================================
# 🚘 MAIN ANALYSIS
# ==============================================
def analyze(fullcar, body, wheel1, wheel2, engine):
    vehicle = extract_vehicle_details(fullcar)
    year = infer_model_year(vehicle, fullcar)

    parts = {
        "Full Car": fullcar,
        "Body": body,
        "Wheel 1": wheel1,
        "Wheel 2": wheel2,
        "Engine": engine
    }

    ratings, html = [], ""
    for name, img in parts.items():
        if img:
            r, desc = inspect_part(name, enhance(img))
            ratings.append(r)
            html += f"<h4>{name} — {r}/10 {emoji(r)}</h4><p>{desc}</p>"

    avg = sum(ratings)/len(ratings)

    return f"""
    <h2>🚗 Vehicle Summary</h2>
    <ul>
      <li><b>Type:</b> {vehicle.get('Type')}</li>
      <li><b>Make:</b> {vehicle.get('Make')}</li>
      <li><b>Model:</b> {vehicle.get('Model')}</li>
      <li><b>Color:</b> {vehicle.get('Color')}</li>
      <li><b>Estimated Year:</b> {year}</li>
    </ul>
    <h2>⭐ Average Rating: {avg:.1f} {emoji(avg)}</h2>
    <hr>{html}
    """

# ==============================================
# 🎨 GRADIO UI
# ==============================================
with gr.Blocks() as demo:
    gr.Markdown("## 🚘 Used Car Inspection (Gemini + RAG)")
    with gr.Row():
        fullcar = gr.Image(label="Full Car", type="pil")
        body = gr.Image(label="Body", type="pil")
    with gr.Row():
        wheel1 = gr.Image(label="Wheel 1", type="pil")
        wheel2 = gr.Image(label="Wheel 2", type="pil")
    engine = gr.Image(label="Engine", type="pil")
    btn = gr.Button("Run Inspection")
    out = gr.HTML()
    btn.click(analyze, [fullcar, body, wheel1, wheel2, engine], out)

demo.launch(share=True)

SecretNotFoundError: Secret GOOGLE_API_KEY does not exist.